In [1]:
!git clone -b master https://github.com/nataliepham6720/legible_autonomous_driving

Cloning into 'legible_autonomous_driving'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 130 (delta 47), reused 55 (delta 24), pack-reused 45 (from 2)
Receiving objects: 100% (130/130), 301.29 MiB | 41.47 MiB/s, done.
Resolving deltas: 100% (52/52), done.
Updating files: 100% (35/35), done.


In [5]:
%cd /content/legible_autonomous_driving

/content/legible_autonomous_driving


In [3]:
!tar -xzvf data/vqa_train_10k.tar.gz -C data/
!tar -xzvf data/vqa_test_1k.tar.gz -C data/

vqa_train_10k.pkl
vqa_test_1k.pkl


In [10]:
# # Legible AD — Evaluation Table
#
# Produces the three-metric comparison table for four model configurations:
#
# |              Row               |              Checkpoint            | model_type |
# |--------------------------------|------------------------------------|------------|
# | LLaMA2 (base)                  | none — fresh LoRA (raw LLaMA-2-7B) | vanilla    |
# | LLM-Driver (1)                 | `stage1_pretrained_model/`         | vanilla    |
# | LLM-Driver (2 w/ pretrained)   | `stage2_with_pretrained/`          | vanilla    |
# | LLM-Driver (2 w/o pretrained)  | `stage2_without_pretrained/`       | vanilla    |
#
# **Legibility note:**
# The existing stage-2 checkpoints (`adapter_model.bin`) contain LoRA +
# `vector_encoder` + `llm_proj` weights, but **no `route_head` /
# `route_queries`** (these live in `VectorRouteGuard`, the outer wrapper,
# and were not saved at training time). Without them the evaluator falls
# back to the ground-truth route for legibility — so the *legibility score
# will be identical* across all four rows (it reflects the scene's inherent
# route legibility, not any model choice). Cosine similarity and scene
# accuracy differ because they come from the LLM's text generation.
#
# **GPU requirement:** ≥ 20 GB VRAM (A100 recommended).

import subprocess, sys

def _run(cmd, **kw):
    r = subprocess.run(cmd, shell=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")
    return r

_run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

# ## 1 · Install dependencies
_run(
    "pip install -q "
    "torch>=2.0.0 "
    "transformers==4.35.2 "
    "peft==0.6.2 "
    "accelerate==0.24.1 "
    "bitsandbytes "
    "datasets==4.4.2 "
    "einops==0.4.1 "
    "sentence-transformers "
    "safetensors "
    "fire "
    "Pillow "
    "tqdm "
    "wandb"
)

# ## 1b · bitsandbytes / peft compatibility patch
# peft==0.6.2 directly reads `MatmulLtState.memory_efficient_backward`,
# which bitsandbytes removed in 0.42.0. If Colab's runtime has a newer
# bitsandbytes cached despite the pin above, this patch restores the
# missing attribute so LoRA injection succeeds.

import importlib

def _patch_bnb_matmul_state():
    for _path in ("bitsandbytes.autograd._functions", "bitsandbytes.functional"):
        try:
            _mod = importlib.import_module(_path)
            _cls = getattr(_mod, "MatmulLtState", None)
            if _cls is not None:
                if not hasattr(_cls, "memory_efficient_backward"):
                    _cls.memory_efficient_backward = False
                    print(f"[bnb patch] Added memory_efficient_backward=False "
                          f"to {_path}.MatmulLtState")
                else:
                    print(f"[bnb patch] memory_efficient_backward already present — no patch needed")
                return
        except ModuleNotFoundError:
            continue
    print("[bnb patch] MatmulLtState not found; patch skipped")

_patch_bnb_matmul_state()

[bnb patch] Added memory_efficient_backward=False to bitsandbytes.autograd._functions.MatmulLtState


In [ ]:
import os, sys

REPO_DIR  = "/content/legible_autonomous_driving"
HF_TOKEN  = "hf_xxx"    # HuggingFace token for LLaMA-2

if not os.path.isdir(REPO_DIR):
    _run(
        "git clone https://github.com/nataliepham6720/legible_autonomous_driving.git "
        f"{REPO_DIR}"
    )

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

if HF_TOKEN:
    _run(f"huggingface-cli login --token {HF_TOKEN}")

# ## 3 · Prepare test data

# %%
TEST_PKL = "data/vqa_test_1k.pkl"
if not os.path.exists(TEST_PKL):
    _run("tar -xzvf data/vqa_test_1k.tar.gz -C data/")

# Need a small training split so get_train_val_data() can return a val set.
TRAIN_PKL = "data/vqa_train_10k.pkl"
if not os.path.exists(TRAIN_PKL):
    _run("tar -xzvf data/vqa_train_10k.tar.gz -C data/")


### Evaluation

In [48]:
# ## 4 · Evaluation helpers
import gc
import numpy as np
import torch
import torch.nn.functional as F

from utils.model_utils import load_llama_tokenizer, load_model
from utils.training_utils import get_train_val_data
from train_legible import VectorRouteGuard, patch_vector_fields
from eval_legible import (
    SimpleEmbedder,
    evaluate_legibility_behavior,
    set_global_seed,
    select_pedestrian_samples,
)

BASE_MODEL  = "meta-llama/Llama-2-7b-hf"
N_SAMPLES   = 50    # pedestrian-containing samples per model; increase for tighter CIs
EVAL_SEED   = 2026
LORA_R      = 16


def _load_vanilla_with_custom_modules(ckpt_path: str, lora_r: int = 16):
    """
    Load a VectorLMWithLoRA from a LoRA checkpoint and restore
    vector_encoder / llm_proj from adapter_model.bin if present
    (they are NOT loaded by PEFT's set_peft_model_state_dict).
    Then wrap in VectorRouteGuard.
    """
    base = load_model(
        base_model=BASE_MODEL,
        resume_from_checkpoint=ckpt_path,
        lora_r=lora_r,
    ).cuda()

    # Manually restore vector_encoder / llm_proj if they live in adapter_model.bin
    adapter_bin = os.path.join(ckpt_path, "adapter_model.bin") if ckpt_path else None
    if adapter_bin and os.path.isfile(adapter_bin):
        ckpt = torch.load(adapter_bin, map_location="cpu")
        CUSTOM_PREFIXES = ("vector_encoder", "llm_proj")
        custom_weights = {
            k: v for k, v in ckpt.items()
            if any(k.startswith(p) for p in CUSTOM_PREFIXES)
        }
        if custom_weights:
            missing, unexpected = base.load_state_dict(custom_weights, strict=False)
            loaded = [k for k in custom_weights if k not in missing]
            print(f"  Manually restored {len(loaded)} custom module tensors "
                  f"(vector_encoder / llm_proj) from adapter_model.bin")

    model = VectorRouteGuard(base).cuda()
    model.eval()
    return model


def _free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [14]:
# ## 5 · Model registry
# Empty string → raw LLaMA-2 base (LoRA B=0, contribution = 0).

MODELS = [
    ("LLaMA2 (base model)",             ""),
    ("LLM-Driver (1)",                  "models/weights/stage1_pretrained_model/"),
    ("LLM-Driver (2 w/ pretrained)",    "models/weights/stage2_with_pretrained/"),
    ("LLM-Driver (2 w/o pretrained)",   "models/weights/stage2_without_pretrained/"),
]

In [16]:
# ## 6 · Run evaluations
tokenizer  = load_llama_tokenizer(BASE_MODEL)

# Val data is used by evaluate_legibility_behavior to select samples with pedestrians.
_, val_data = get_train_val_data(TEST_PKL, tokenizer, val_set_size=100)
val_data    = patch_vector_fields(val_data)

set_global_seed(EVAL_SEED)

results = {}

for display_name, ckpt_path in MODELS:
    print(f"\n{'='*65}")
    print(f"  Model : {display_name}")
    print(f"  Ckpt  : {ckpt_path or '(none — LLaMA-2 base)'}")
    print(f"{'='*65}\n")

    _free_gpu()

    model = _load_vanilla_with_custom_modules(ckpt_path, lora_r=LORA_R)

    metrics = evaluate_legibility_behavior(
        model,
        tokenizer,
        val_data,
        model_type="vanilla",
        n_samples=N_SAMPLES,
        seed=EVAL_SEED,
    )

    results[display_name] = metrics
    print(f"\n  → legibility={metrics['legibility']:.4f}  "
          f"cosine={metrics['cosine_similarity']:.4f}  "
          f"scene_acc={metrics['scene_accuracy']:.4f}")

    # Free GPU before loading next model
    del model
    _free_gpu()



Augment times: 100%|██████████| 1/1 [00:06<00:00,  6.76s/it]


Map (num_proc=8):   0%|          | 0/1880 [00:00<?, ? examples/s]

Processing val (num_proc=1):   0%|          | 0/100 [00:00<?, ? examples/s]

Validating dataset vector fields...
Dataset sanitized

  Model : LLaMA2 (base model)
  Ckpt  : (none — LLaMA-2 base)



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

Some weights of LlamaForCausalLMVectorInput were not initialized from the model checkpoint at meta-llama/Llama-2-7b-hf and are newly initialized: ['weighted_mask']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Restarting from models/weights/stage2_with_pretrained/adapter_model.bin
  Manually restored 122 custom module tensors (vector_encoder / llm_proj) from adapter_model.bin

================ EVALUATION ================


================ SAMPLE 1/50 ================

Route type: ground_truth
Route (first 5 points):
tensor([[ 4.4936, -3.9962],
        [-0.2960, -4.5209],
        [ 4.9433,  0.6039],
        [ 2.9162, -4.7194],
        [-0.4258, -0.9528]], device='cuda:0')
Route L2 Error (normalized): 0.0000

Legibility per agent:
  Vehicle 0: pred=0.0354  gt=0.0354
  Pedestrian 0 (cross=0): pred=0.5420  gt=0.5420

Scene Legibility (pred route): 0.2887
Scene Legibility (GT route):   0.2887

Evaluating QA for sample 1...

Q [distance]: What is the distance of the farthest pedestrian?
Pred: The answer to this question depends on how you define "far." If we assume that a person walking at an average pace will take about one second per meter, then it would be reasonable for us to say that any obj

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForCausalLMVectorInput were not initialized from the model checkpoint at meta-llama/Llama-2-7b-hf and are newly initialized: ['weighted_mask']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Restarting from models/weights/stage2_without_pretrained/adapter_model.bin
  Manually restored 122 custom module tensors (vector_encoder / llm_proj) from adapter_model.bin

================ EVALUATION ================


================ SAMPLE 1/50 ================

Route type: ground_truth
Route (first 5 points):
tensor([[ 4.4936, -3.9962],
        [-0.2960, -4.5209],
        [ 4.9433,  0.6039],
        [ 2.9162, -4.7194],
        [-0.4258, -0.9528]], device='cuda:0')
Route L2 Error (normalized): 0.0000

Legibility per agent:
  Vehicle 0: pred=0.0354  gt=0.0354
  Pedestrian 0 (cross=0): pred=0.5420  gt=0.5420

Scene Legibility (pred route): 0.2887
Scene Legibility (GT route):   0.2887

Evaluating QA for sample 1...

Q [distance]: What is the distance of the farthest pedestrian?
Pred: The answer to this question can be found by looking at the image above and counting how many cars are between you (the driver) and that person, then subtracting one from it because there's only a single c

In [17]:
# ## 7 · Results table
import pandas as pd

rows = []
for name, m in results.items():
    rows.append({
        "Model":              name,
        "Legible score":      round(m["legibility"],        6),
        "Cosine similarity":  round(m["cosine_similarity"], 6),
        "Scene accuracy":     round(m["scene_accuracy"],    6),
    })

df = pd.DataFrame(rows)
df = df.set_index("Model")

print("\n" + "="*65)
print("EVALUATION RESULTS")
print("="*65)
print(df.to_string())
print("="*65)

# Markdown-friendly output
print("\n### Markdown table\n")
print(df.to_markdown())


EVALUATION RESULTS
                               Legible score  Cosine similarity  Scene accuracy
Model                                                                          
LLaMA2 (base model)                   0.3189             0.0507          0.5018
LLM-Driver (1)                        0.3189             0.0635          0.5224
LLM-Driver (2 w/ pretrained)          0.3189             0.0650          0.5306
LLM-Driver (2 w/o pretrained)         0.3189             0.0561          0.5344

### Markdown table

| Model                         |   Legible score |   Cosine similarity |   Scene accuracy |
|:------------------------------|----------------:|--------------------:|-----------------:|
| LLaMA2 (base model)           |          0.3189 |              0.0507 |           0.5018 |
| LLM-Driver (1)                |          0.3189 |              0.0635 |           0.5224 |
| LLM-Driver (2 w/ pretrained)  |          0.3189 |              0.065  |           0.5306 |
| LLM-Driver (